In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [ ]:
# load from target/artifact_sizes.json (generated by cargo bench)
import json
from pathlib import Path

records = json.loads(Path('../target/artifact_sizes.json').read_text())
df = pd.DataFrame(records)
df['R'] = pd.to_numeric(df['R'], errors='coerce')
df

In [ ]:
# R=0 separately since mu_uv is empty so t-coefficient halves
df_zero = df[df['R'] == 0]
m0 = LinearRegression().fit(df_zero[['t']], df_zero['size'])
print(f"R=0: size ≈ {m0.intercept_:.1f} + {m0.coef_[0]:.1f}·t")
print(f"R^2 = {m0.score(df_zero[['t']], df_zero['size']):.8f}")

In [ ]:
# R>0
df_pos = df[df['R'] > 0]
model = LinearRegression().fit(df_pos[['R', 't']], df_pos['size'])

print(f"size ≈ {model.intercept_:.1f} + {model.coef_[0]:.1f}·R + {model.coef_[1]:.1f}·t")
print(f"R^2 = {model.score(df_pos[['R','t']], df_pos['size']):.8f}")

In [ ]:
df_lhs = df[df['scheme'] == 'mklhs'].dropna(subset=['t'])
model_lhs = LinearRegression().fit(df_lhs[['t']], df_lhs['size'])
print(f"mklhs size ≈ {model_lhs.intercept_:.1f} + {model_lhs.coef_[0]:.1f}·t")
print(f"R^2 = {model_lhs.score(df_lhs[['t']], df_lhs['size']):.8f} ")

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# fitted plane (R > 0 model)
R_grid, t_grid = np.meshgrid(np.linspace(0, 16, 50), np.linspace(2, 10, 50))
size_grid = model.intercept_ + model.coef_[0] * R_grid + model.coef_[1] * t_grid
ax.plot_surface(R_grid, t_grid, size_grid, alpha=0.3, color='steelblue')

ax.scatter(df['R'], df['t'], df['size'], color='red', zorder=5)

ax.set(xlabel='R (rank)', ylabel='t (signers)', zlabel='bytes')
ax.set_title(f'mkqhs eval sig size\n≈ {model.intercept_:.0f} + {model.coef_[0]:.0f}·R + {model.coef_[1]:.0f}·t')
plt.tight_layout()
plt.show()